# TRADD — Cleaned Research Notebook

This notebook keeps the final 500-video FaceForensics++ pipeline, TRADD RGB/DCT experiments, temporal GRU, and reliability evaluation. Earlier prototype experiments, repeated download commands, and repository housekeeping cells have been excluded.

**Note:** Run cells in order. Dataset and checkpoint paths remain the original Google Drive paths.


## Environment and project configuration


In [ ]:
import sys
import torch
import torchvision
import cv2
import numpy as np

print("===== TRADD ENVIRONMENT CHECK =====")

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)

print("\n===== HARDWARE CHECK =====")

if torch.cuda.is_available():
    print("GPU Available:", True)
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("CUDA Version:", torch.version.cuda)
else:
    print("GPU Available: False")
    print("Running on CPU")

print("\nEnvironment check completed.")

In [ ]:
%pip install -q pandas matplotlib seaborn scikit-learn tqdm pyyaml tensorboard

In [ ]:
import torch
import torchvision
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import yaml
from tqdm.auto import tqdm
from torch.utils.tensorboard import SummaryWriter

print("All required libraries imported successfully!")

print("\nPyTorch:", torch.__version__)
print("GPU:", torch.cuda.is_available())
print("OpenCV:", cv2.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Pandas:", pd.__version__)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/TRADD")

folders = [
    "datasets/raw",
    "datasets/processed",
    "datasets/train",
    "datasets/validation",
    "datasets/test",
    "models",
    "preprocessing",
    "temporal",
    "reliability",
    "deployment",
    "evaluation",
    "experiments",
    "app",
    "configs",
    "results/plots",
    "results/logs",
    "results/reports",
    "notebooks",
    "tests"
]

for folder in folders:
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)

print("TRADD project directories created successfully!")
print("Project location:", PROJECT_ROOT)

In [ ]:
config = """
project:
  name: TRADD
  description: Temporal Reliability-Aware Deepfake Detection

dataset:
  name: FaceForensics++
  image_size: 224
  batch_size: 32
  num_workers: 2

model:
  architecture: mobilenet_v3_small
  num_classes: 2
  pretrained: true

training:
  epochs: 10
  learning_rate: 0.0001
  weight_decay: 0.0001

temporal:
  window_size: 30
  fake_threshold: 0.70
  uncertainty_threshold: 0.30

device:
  use_cuda: true
"""

config_path = PROJECT_ROOT / "configs" / "config.yaml"
config_path.write_text(config.strip())

print("Configuration file created:", config_path)

## FaceForensics++ dataset preparation and identity-aware split


In [ ]:
import os
import glob

BASE = "/content/drive/MyDrive/TRADD/datasets/raw/FaceForensicsPP_expansion"

REAL_DIR = os.path.join(
    BASE, "original_sequences/youtube/c23/videos"
)

METHODS = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]

# Collect original video IDs
real_files = glob.glob(os.path.join(REAL_DIR, "*.mp4"))
real_ids = {os.path.splitext(os.path.basename(f))[0] for f in real_files}

print("Original videos:", len(real_ids))

# Check manipulated video IDs
for method in METHODS:

    fake_dir = os.path.join(
        BASE, "manipulated_sequences", method, "c23", "videos"
    )

    fake_files = glob.glob(os.path.join(fake_dir, "*.mp4"))

    matched = 0
    unmatched = []

    for file in fake_files:
        filename = os.path.splitext(os.path.basename(file))[0]
        parts = filename.split("_")

        if len(parts) != 2:
            unmatched.append(filename)
            continue

        target_id, source_id = parts

        if target_id in real_ids and source_id in real_ids:
            matched += 1
        else:
            unmatched.append(filename)

    print(f"\n{method}")
    print("Total videos:", len(fake_files))
    print("Matched videos:", matched)
    print("Unmatched videos:", len(unmatched))
    print("Examples:", unmatched[:5])

In [ ]:
import os
import glob
from collections import defaultdict

BASE = "/content/drive/MyDrive/TRADD/datasets/raw/FaceForensicsPP_expansion"

REAL_DIR = os.path.join(
    BASE, "original_sequences/youtube/c23/videos"
)

METHODS = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]

# Create graph
graph = defaultdict(set)

# Add all original video IDs
real_files = glob.glob(os.path.join(REAL_DIR, "*.mp4"))
real_ids = {
    os.path.splitext(os.path.basename(f))[0]
    for f in real_files
}

for video_id in real_ids:
    graph[video_id]

# Connect identities appearing together in manipulated videos
for method in METHODS:

    fake_dir = os.path.join(
        BASE, "manipulated_sequences", method, "c23", "videos"
    )

    for file in glob.glob(os.path.join(fake_dir, "*.mp4")):

        filename = os.path.splitext(os.path.basename(file))[0]
        target_id, source_id = filename.split("_")

        graph[target_id].add(source_id)
        graph[source_id].add(target_id)

# Find connected components
visited = set()
components = []

for node in graph:

    if node in visited:
        continue

    stack = [node]
    component = []

    while stack:

        current = stack.pop()

        if current in visited:
            continue

        visited.add(current)
        component.append(current)

        for neighbour in graph[current]:
            if neighbour not in visited:
                stack.append(neighbour)

    components.append(component)

components.sort(key=len, reverse=True)

print("Total identity groups:", len(components))
print("\nGroup sizes:")

for i, group in enumerate(components, 1):
    print(f"Group {i}: {len(group)} identities")

print("\nLargest group size:", len(components[0]))

In [ ]:
import os
import glob
import random
import pandas as pd
from collections import defaultdict

BASE = "/content/drive/MyDrive/TRADD/datasets/raw/FaceForensicsPP_expansion"

REAL_DIR = os.path.join(
    BASE, "original_sequences/youtube/c23/videos"
)

METHODS = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]

# 1. Rebuild identity graph
real_files = glob.glob(os.path.join(REAL_DIR, "*.mp4"))

real_ids = {
    os.path.splitext(os.path.basename(f))[0]
    for f in real_files
}

graph = defaultdict(set)

for vid in real_ids:
    graph[vid]

for method in METHODS:
    fake_dir = os.path.join(
        BASE, "manipulated_sequences", method, "c23", "videos"
    )

    for path in glob.glob(os.path.join(fake_dir, "*.mp4")):
        name = os.path.splitext(os.path.basename(path))[0]
        target, source = name.split("_")

        graph[target].add(source)
        graph[source].add(target)

# 2. Find connected components
visited = set()
groups = []

for node in graph:
    if node in visited:
        continue

    stack = [node]
    group = []

    while stack:
        current = stack.pop()

        if current in visited:
            continue

        visited.add(current)
        group.append(current)

        for neighbour in graph[current]:
            if neighbour not in visited:
                stack.append(neighbour)

    groups.append(sorted(group))

# 3. Shuffle and assign groups
random.seed(42)
random.shuffle(groups)

train_groups = groups[:30]
val_groups = groups[30:40]
test_groups = groups[40:50]

group_to_split = {}

for split, selected_groups in [
    ("train", train_groups),
    ("val", val_groups),
    ("test", test_groups)
]:
    for group in selected_groups:
        for identity in group:
            group_to_split[identity] = split

# 4. Build video manifest
records = []

# Original videos
for path in real_files:
    vid = os.path.splitext(os.path.basename(path))[0]

    records.append({
        "video_name": vid + ".mp4",
        "video_path": path,
        "label": 0,
        "class": "REAL",
        "manipulation": "Original",
        "identity_group": next(
            i for i, g in enumerate(groups) if vid in g
        ),
        "split": group_to_split[vid]
    })

# Manipulated videos
for method in METHODS:
    fake_dir = os.path.join(
        BASE, "manipulated_sequences", method, "c23", "videos"
    )

    for path in glob.glob(os.path.join(fake_dir, "*.mp4")):
        name = os.path.splitext(os.path.basename(path))[0]
        target, source = name.split("_")

        assert group_to_split[target] == group_to_split[source]

        records.append({
            "video_name": name + ".mp4",
            "video_path": path,
            "label": 1,
            "class": "FAKE",
            "manipulation": method,
            "identity_group": next(
                i for i, g in enumerate(groups) if target in g
            ),
            "split": group_to_split[target]
        })

# 5. Save CSV
manifest = pd.DataFrame(records)

OUTPUT = os.path.join(BASE, "video_split_manifest.csv")
manifest.to_csv(OUTPUT, index=False)

# 6. Verify results
print("Total videos:", len(manifest))
print("\nSplit distribution:")
print(pd.crosstab(manifest["split"], manifest["class"]))

print("\nManipulation distribution:")
print(pd.crosstab(manifest["split"], manifest["manipulation"]))

print("\nSaved manifest:", OUTPUT)

# Verify no identity group crosses splits
assert manifest.groupby("identity_group")["split"].nunique().max() == 1

print("\nIdentity leakage check: PASSED")

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

BASE = "/content/drive/MyDrive/TRADD/datasets/raw/FaceForensicsPP_expansion"

MANIFEST_PATH = os.path.join(BASE, "video_split_manifest.csv")

FRAME_DIR = (
    "/content/drive/MyDrive/TRADD/datasets/processed/frames_expansion"
)

os.makedirs(FRAME_DIR, exist_ok=True)

manifest = pd.read_csv(MANIFEST_PATH)

print("Videos to process:", len(manifest))
print(manifest.head())

In [ ]:
frame_records = []
failed_videos = []

FRAMES_PER_VIDEO = 10

for _, row in tqdm(manifest.iterrows(), total=len(manifest)):

    video_path = row["video_path"]

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        failed_videos.append(video_path)
        continue

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    if total_frames <= 0:
        failed_videos.append(video_path)
        cap.release()
        continue

    # Select evenly spaced frame positions
    frame_indices = np.linspace(
        0,
        total_frames - 1,
        FRAMES_PER_VIDEO,
        dtype=int
    )

    video_id = os.path.splitext(row["video_name"])[0]

    save_dir = os.path.join(
        FRAME_DIR,
        row["split"],
        row["class"].lower(),
        row["manipulation"],
        video_id
    )

    os.makedirs(save_dir, exist_ok=True)

    for frame_number in frame_indices:

        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_number))
        success, frame = cap.read()

        if not success:
            continue

        frame_name = f"frame_{frame_number:06d}.jpg"
        frame_path = os.path.join(save_dir, frame_name)

        saved = cv2.imwrite(
            frame_path,
            frame,
            [cv2.IMWRITE_JPEG_QUALITY, 95]
        )

        if not saved:
            continue

        frame_records.append({
            "video_name": row["video_name"],
            "label": row["label"],
            "class": row["class"],
            "manipulation": row["manipulation"],
            "split": row["split"],
            "identity_group": row["identity_group"],
            "frame_number": int(frame_number),
            "fps": fps,
            "total_frames": total_frames,
            "frame_path": frame_path
        })

    cap.release()

# Save frame metadata
frame_df = pd.DataFrame(frame_records)

OUTPUT_CSV = os.path.join(FRAME_DIR, "frame_metadata.csv")
frame_df.to_csv(OUTPUT_CSV, index=False)

print("\nFRAME EXTRACTION COMPLETE")
print("Extracted frames:", len(frame_df))
print("Failed videos:", len(failed_videos))
print("Metadata saved:", OUTPUT_CSV)

print("\nFrames by split and class:")
print(pd.crosstab(frame_df["split"], frame_df["class"]))

print("\nFrames by manipulation:")
print(pd.crosstab(frame_df["split"], frame_df["manipulation"]))

In [ ]:
import os

ROOT = "/content/drive/MyDrive/TRADD"

for folder, dirs, files in os.walk(ROOT):
    for file in files:
        if file in ["video_split_manifest.csv", "frame_metadata.csv"]:
            print(os.path.join(folder, file))

In [ ]:
YUNET_PATH = os.path.join(ROOT, "models", "face_detection_yunet.onnx")

print("YuNet exists:", os.path.exists(YUNET_PATH))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

ROOT = "/content/drive/MyDrive/TRADD"

BASE = ROOT + "/datasets/raw/FaceForensicsPP_expansion"

FRAME_DIR = ROOT + "/datasets/processed/frames_expansion"

print("Manifest:", os.path.exists(BASE + "/video_split_manifest.csv"))
print("Frame metadata:", os.path.exists(FRAME_DIR + "/frame_metadata.csv"))
print("Frame folder:", os.path.exists(FRAME_DIR))
print("YuNet model:", os.path.exists(ROOT + "/models/face_detection_yunet.onnx"))

In [ ]:
import os
import multiprocessing

print("CPU cores:", multiprocessing.cpu_count())
print("OpenCV threads:", cv2.getNumThreads())

In [ ]:
import os
import cv2
import pandas as pd
from tqdm.auto import tqdm

ROOT = "/content/drive/MyDrive/TRADD"
FRAME_DIR = ROOT + "/datasets/processed/frames_expansion"
FACE_DIR = ROOT + "/datasets/processed/faces_expansion"
YUNET_PATH = ROOT + "/models/face_detection_yunet.onnx"

metadata = pd.read_csv(FRAME_DIR + "/frame_metadata.csv")

detector = cv2.FaceDetectorYN.create(
    YUNET_PATH, "", (640, 640), 0.7, 0.3, 5000
)

cv2.setNumThreads(1)
os.makedirs(FACE_DIR, exist_ok=True)

records = []
failed = []

for _, row in tqdm(metadata.iterrows(), total=len(metadata)):

    relative_path = os.path.relpath(row["frame_path"], FRAME_DIR)
    face_path = os.path.join(
        FACE_DIR, os.path.splitext(relative_path)[0] + ".jpg"
    )

    # Reuse an existing valid crop
    if os.path.exists(face_path):
        existing = cv2.imread(face_path)
        if existing is not None and existing.shape[:2] == (224, 224):
            records.append({**row.to_dict(), "face_path": face_path})
            continue

    image = cv2.imread(row["frame_path"])

    if image is None:
        failed.append(row["frame_path"])
        continue

    h, w = image.shape[:2]

    # Resize only for face detection
    scale = min(640 / max(h, w), 1.0)
    small = cv2.resize(
        image, (int(w * scale), int(h * scale))
    )

    sh, sw = small.shape[:2]
    detector.setInputSize((sw, sh))

    _, faces = detector.detect(small)

    if faces is None or len(faces) == 0:
        failed.append(row["frame_path"])
        continue

    # Select largest face and map coordinates to original image
    face = max(faces, key=lambda f: f[2] * f[3])

    x, y, fw, fh = face[:4]

    x, fw = x / scale, fw / scale
    y, fh = y / scale, fh / scale

    cx, cy = x + fw / 2, y + fh / 2
    side = max(fw, fh) * 1.5

    x1 = max(0, int(cx - side / 2))
    y1 = max(0, int(cy - side / 2))
    x2 = min(w, int(cx + side / 2))
    y2 = min(h, int(cy + side / 2))

    crop = image[y1:y2, x1:x2]

    if crop.size == 0:
        failed.append(row["frame_path"])
        continue

    crop = cv2.resize(crop, (224, 224))

    os.makedirs(os.path.dirname(face_path), exist_ok=True)

    if not cv2.imwrite(face_path, crop):
        failed.append(row["frame_path"])
        continue

    records.append({**row.to_dict(), "face_path": face_path})

face_df = pd.DataFrame(records)

face_df.to_csv(
    FACE_DIR + "/face_metadata.csv", index=False
)

print("Successfully processed:", len(face_df))
print("Failed:", len(failed))
print("Metadata saved successfully.")

> Original cell 95 was empty or unavailable in the exported source.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import random

ROOT = "/content/drive/MyDrive/TRADD"

FACE_DIR = ROOT + "/datasets/processed/faces_expansion"

metadata = pd.read_csv(FACE_DIR + "/face_metadata.csv")

print("Total face crops:", len(metadata))
print("Missing crops:", sum(
    not os.path.exists(p) for p in metadata["face_path"]
))
print("\nDistribution:")
print(metadata.groupby(["split", "class"]).size())

In [ ]:
samples = metadata.sample(8, random_state=42)

plt.figure(figsize=(12, 8))

for i, (_, row) in enumerate(samples.iterrows()):

    image = cv2.imread(row["face_path"])
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    plt.subplot(2, 4, i + 1)
    plt.imshow(image)
    plt.title(
        f'{row["class"]} | {row["manipulation"]}',
        fontsize=9
    )
    plt.axis("off")

plt.tight_layout()
plt.show()

## TRADD architecture and RGB / DCT experiments


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

# Create an orthonormal 8x8 DCT transformation matrix
def create_dct_matrix(N=8):
    matrix = torch.zeros(N, N)

    for k in range(N):
        alpha = (1 / N) ** 0.5 if k == 0 else (2 / N) ** 0.5

        for n in range(N):
            matrix[k, n] = alpha * torch.cos(
                torch.tensor(
                    torch.pi * (2 * n + 1) * k / (2 * N)
                )
            )

    return matrix


class TRADD(nn.Module):

    def __init__(self, num_classes=2):

        super().__init__()

        # 1. Spatial branch: pretrained MobileNetV3 Small
        backbone = models.mobilenet_v3_small(
            weights=models.MobileNet_V3_Small_Weights.DEFAULT
        )

        self.spatial_features = backbone.features
        self.spatial_pool = backbone.avgpool

        # 2. ImageNet normalization values
        self.register_buffer(
            "mean",
            torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        )

        self.register_buffer(
            "std",
            torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        )

        # 3. DCT transformation matrix
        self.register_buffer("dct_matrix", create_dct_matrix())

        # 4. Frequency branch
        self.frequency_branch = nn.Sequential(
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1)
        )

        # 5. Feature fusion and classification
        self.classifier = nn.Sequential(
            nn.Linear(576 + 128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def extract_frequency(self, x):

        # Reverse ImageNet normalization
        x = (x * self.std + self.mean).clamp(0, 1)

        # Convert RGB image to grayscale
        gray = (
            0.299 * x[:, 0:1] +
            0.587 * x[:, 1:2] +
            0.114 * x[:, 2:3]
        )

        # Divide image into non-overlapping 8x8 blocks
        blocks = F.unfold(
            gray,
            kernel_size=8,
            stride=8
        )

        batch_size = x.shape[0]

        blocks = blocks.transpose(1, 2).reshape(
            batch_size, -1, 8, 8
        )

        # Apply 2D DCT to every block
        dct = torch.einsum(
            "ij,bnjk,lk->bnil",
            self.dct_matrix,
            blocks,
            self.dct_matrix
        )

        # Arrange coefficients as 64 feature maps
        frequency = dct.reshape(
            batch_size, -1, 64
        ).transpose(1, 2).reshape(
            batch_size, 64, 28, 28
        )

        # Compress the coefficient range while preserving signs
        frequency = (
            torch.sign(frequency) *
            torch.log1p(torch.abs(frequency))
        )

        return frequency

    def forward(self, x):

        # Spatial features
        spatial = self.spatial_features(x)
        spatial = self.spatial_pool(spatial)
        spatial = torch.flatten(spatial, 1)

        # Frequency features
        frequency = self.extract_frequency(x)
        frequency = self.frequency_branch(frequency)
        frequency = torch.flatten(frequency, 1)

        # Combine both feature representations
        combined = torch.cat(
            [spatial, frequency], dim=1
        )

        # REAL / FAKE prediction
        output = self.classifier(combined)

        return output


# Initialize model
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = TRADD().to(device)

print("Model initialized successfully!")
print("Device:", device)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

# Create an orthonormal 8x8 DCT transformation matrix
def create_dct_matrix(N=8):
    matrix = torch.zeros(N, N)

    for k in range(N):
        alpha = (1 / N) ** 0.5 if k == 0 else (2 / N) ** 0.5

        for n in range(N):
            matrix[k, n] = alpha * torch.cos(
                torch.tensor(
                    torch.pi * (2 * n + 1) * k / (2 * N)
                )
            )

    return matrix


class TRADD(nn.Module):

    def __init__(self, num_classes=2):

        super().__init__()

        # 1. Spatial branch: pretrained MobileNetV3 Small
        backbone = models.mobilenet_v3_small(
            weights=models.MobileNet_V3_Small_Weights.DEFAULT
        )

        self.spatial_features = backbone.features
        self.spatial_pool = backbone.avgpool

        # 2. ImageNet normalization values
        self.register_buffer(
            "mean",
            torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        )

        self.register_buffer(
            "std",
            torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        )

        # 3. DCT transformation matrix
        self.register_buffer("dct_matrix", create_dct_matrix())

        # 4. Frequency branch
        self.frequency_branch = nn.Sequential(
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1)
        )

        # 5. Feature fusion and classification
        self.classifier = nn.Sequential(
            nn.Linear(576 + 128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def extract_frequency(self, x):

        # Reverse ImageNet normalization
        x = (x * self.std + self.mean).clamp(0, 1)

        # Convert RGB image to grayscale
        gray = (
            0.299 * x[:, 0:1] +
            0.587 * x[:, 1:2] +
            0.114 * x[:, 2:3]
        )

        # Divide image into non-overlapping 8x8 blocks
        blocks = F.unfold(
            gray,
            kernel_size=8,
            stride=8
        )

        batch_size = x.shape[0]

        blocks = blocks.transpose(1, 2).reshape(
            batch_size, -1, 8, 8
        )

        # Apply 2D DCT to every block
        dct = torch.einsum(
            "ij,bnjk,lk->bnil",
            self.dct_matrix,
            blocks,
            self.dct_matrix
        )

        # Arrange coefficients as 64 feature maps
        frequency = dct.reshape(
            batch_size, -1, 64
        ).transpose(1, 2).reshape(
            batch_size, 64, 28, 28
        )

        # Compress the coefficient range while preserving signs
        frequency = (
            torch.sign(frequency) *
            torch.log1p(torch.abs(frequency))
        )

        return frequency

    def forward(self, x):

        # Spatial features
        spatial = self.spatial_features(x)
        spatial = self.spatial_pool(spatial)
        spatial = torch.flatten(spatial, 1)

        # Frequency features
        frequency = self.extract_frequency(x)
        frequency = self.frequency_branch(frequency)
        frequency = torch.flatten(frequency, 1)

        # Combine both feature representations
        combined = torch.cat(
            [spatial, frequency], dim=1
        )

        # REAL / FAKE prediction
        output = self.classifier(combined)

        return output


# Initialize model
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = TRADD().to(device)

print("Model initialized successfully!")
print("Device:", device)

In [ ]:
import os
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

ROOT = "/content/drive/MyDrive/TRADD"
FACE_DIR = ROOT + "/datasets/processed/faces_expansion"

metadata = pd.read_csv(
    FACE_DIR + "/face_metadata.csv"
)

# Image transformations
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.1
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])


class TRADD_Dataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.data = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]

        image = Image.open(row["face_path"]).convert("RGB")
        label = int(row["label"])

        if self.transform:
            image = self.transform(image)

        return image, label


# Separate datasets using existing split column
train_data = metadata[metadata["split"] == "train"]
val_data = metadata[metadata["split"] == "val"]
test_data = metadata[metadata["split"] == "test"]

train_dataset = TRADD_Dataset(train_data, train_transform)
val_dataset = TRADD_Dataset(val_data, eval_transform)
test_dataset = TRADD_Dataset(test_data, eval_transform)

# Create DataLoaders
train_loader = DataLoader(
    train_dataset, batch_size=32,
    shuffle=True, num_workers=2
)

val_loader = DataLoader(
    val_dataset, batch_size=32,
    shuffle=False, num_workers=2
)

test_loader = DataLoader(
    test_dataset, batch_size=32,
    shuffle=False, num_workers=2
)

print("Training frames:", len(train_dataset))
print("Validation frames:", len(val_dataset))
print("Testing frames:", len(test_dataset))
print("DataLoaders recreated successfully!")

In [ ]:
images, labels = next(iter(train_loader))

images = images.to(device)

model.eval()

with torch.no_grad():
    outputs = model(images)

print("Input shape:", images.shape)
print("Output shape:", outputs.shape)
print("Parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
import os
import copy
import torch
import torch.nn as nn
import numpy as np

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

# Training configuration
EPOCHS = 15
PATIENCE = 4
LEARNING_RATE = 0.0001

# Class weights: REAL=0, FAKE=1
class_counts = torch.tensor([600, 2400], dtype=torch.float32)

class_weights = class_counts.sum() / (2 * class_counts)
class_weights = class_weights.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.0001
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

SAVE_DIR = "/content/drive/MyDrive/TRADD/models"
os.makedirs(SAVE_DIR, exist_ok=True)

BEST_PATH = os.path.join(SAVE_DIR, "tradd_spatial_frequency_best.pth")


# One training epoch
def train_one_epoch(model, loader):

    model.train()
    total_loss = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    return total_loss / len(loader.dataset)


# Validation
def evaluate(model, loader):

    model.eval()

    total_loss = 0
    all_labels = []
    all_predictions = []
    all_probabilities = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            probabilities = torch.softmax(outputs, dim=1)
            predictions = outputs.argmax(dim=1)

            total_loss += loss.item() * images.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())
            all_probabilities.extend(
                probabilities[:, 1].cpu().numpy()
            )

    labels = np.array(all_labels)
    predictions = np.array(all_predictions)
    probabilities = np.array(all_probabilities)

    metrics = {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(
            labels, predictions, zero_division=0
        ),
        "recall": recall_score(
            labels, predictions, zero_division=0
        ),
        "f1": f1_score(
            labels, predictions, zero_division=0
        ),
        "auc": roc_auc_score(labels, probabilities)
    }

    return metrics


# Training loop with early stopping
best_val_loss = float("inf")
best_epoch = 0
patience_counter = 0
history = []

for epoch in range(EPOCHS):

    train_loss = train_one_epoch(model, train_loader)
    val_metrics = evaluate(model, val_loader)

    scheduler.step(val_metrics["loss"])

    current_lr = optimizer.param_groups[0]["lr"]

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        **val_metrics
    })

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val Acc: {val_metrics['accuracy']:.4f} | "
        f"Val F1: {val_metrics['f1']:.4f} | "
        f"Val AUC: {val_metrics['auc']:.4f} | "
        f"LR: {current_lr:.6f}"
    )

    # Save best model based on validation loss
    if val_metrics["loss"] < best_val_loss:

        best_val_loss = val_metrics["loss"]
        best_epoch = epoch + 1
        patience_counter = 0

        torch.save({
            "epoch": best_epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": best_val_loss
        }, BEST_PATH)

        print("  Best checkpoint saved.")

    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print("\nEarly stopping triggered.")
        break


print("\nTraining completed.")
print("Best epoch:", best_epoch)
print("Best validation loss:", round(best_val_loss, 4))
print("Checkpoint:", BEST_PATH)

In [ ]:
import torch
import numpy as np

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    balanced_accuracy_score,
    roc_auc_score,
    f1_score
)

# Load best validation-loss checkpoint
checkpoint = torch.load(
    BEST_PATH,
    map_location=device,
    weights_only=False
)

best_loss_model = TRADD().to(device)
best_loss_model.load_state_dict(
    checkpoint["model_state_dict"]
)

# Compare epoch 1 checkpoint and current epoch 5 model
models_to_compare = {
    "Epoch 1 - Best Val Loss": best_loss_model,
    "Epoch 5 - Latest Model": model
}

for name, selected_model in models_to_compare.items():

    selected_model.eval()

    all_labels = []
    all_predictions = []
    all_probabilities = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)

            outputs = selected_model(images)
            probabilities = torch.softmax(outputs, dim=1)

            all_labels.extend(labels.numpy())
            all_predictions.extend(
                outputs.argmax(dim=1).cpu().numpy()
            )
            all_probabilities.extend(
                probabilities[:, 1].cpu().numpy()
            )

    y_true = np.array(all_labels)
    y_pred = np.array(all_predictions)
    y_prob = np.array(all_probabilities)

    print("\n", "=" * 45)
    print(name)
    print("=" * 45)

    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    print("\nBalanced Accuracy:",
          round(balanced_accuracy_score(y_true, y_pred), 4))

    print("Macro F1:",
          round(f1_score(y_true, y_pred, average="macro"), 4))

    print("ROC-AUC:",
          round(roc_auc_score(y_true, y_prob), 4))

    print("\nClassification Report:")
    print(classification_report(
        y_true, y_pred,
        target_names=["REAL", "FAKE"],
        zero_division=0
    ))

In [ ]:
import numpy as np
import torch
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    classification_report
)

# Save the current Epoch 5 model
EPOCH5_PATH = (
    "/content/drive/MyDrive/TRADD/models/"
    "tradd_spatial_frequency_epoch5.pth"
)

torch.save(model.state_dict(), EPOCH5_PATH)

# Collect validation probabilities
model.eval()

y_true = []
y_prob = []

with torch.no_grad():
    for images, labels in val_loader:

        images = images.to(device)
        outputs = model(images)

        probabilities = torch.softmax(outputs, dim=1)

        y_true.extend(labels.numpy())
        y_prob.extend(probabilities[:, 1].cpu().numpy())

y_true = np.array(y_true)
y_prob = np.array(y_prob)

# Search thresholds
thresholds = np.arange(0.30, 0.81, 0.01)

results = []

for threshold in thresholds:

    predictions = (y_prob >= threshold).astype(int)

    score = balanced_accuracy_score(
        y_true, predictions
    )

    results.append((threshold, score))

# Select best threshold
best_threshold, best_score = max(
    results, key=lambda x: x[1]
)

best_predictions = (y_prob >= best_threshold).astype(int)

print("Best threshold:", round(best_threshold, 2))
print("Best balanced accuracy:", round(best_score, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, best_predictions))

print("\nClassification Report:")
print(classification_report(
    y_true,
    best_predictions,
    target_names=["REAL", "FAKE"],
    zero_division=0
))

In [ ]:
import torch
import numpy as np
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    balanced_accuracy_score,
    roc_auc_score,
    f1_score
)

# Load saved Epoch 5 checkpoint
model.load_state_dict(
    torch.load(EPOCH5_PATH, map_location=device, weights_only=True)
)

model.eval()

y_true = []
y_prob = []

with torch.no_grad():
    for images, labels in test_loader:

        images = images.to(device)
        outputs = model(images)

        probabilities = torch.softmax(outputs, dim=1)

        y_true.extend(labels.numpy())
        y_prob.extend(probabilities[:, 1].cpu().numpy())

y_true = np.array(y_true)
y_prob = np.array(y_prob)

# Use the validation-selected threshold
TEST_THRESHOLD = 0.74
y_pred = (y_prob >= TEST_THRESHOLD).astype(int)

print("Test threshold:", TEST_THRESHOLD)
print("Balanced accuracy:",
      balanced_accuracy_score(y_true, y_pred))
print("Macro F1:",
      f1_score(y_true, y_pred, average="macro"))
print("ROC-AUC:", roc_auc_score(y_true, y_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_true, y_pred,
    target_names=["REAL", "FAKE"],
    zero_division=0
))

In [ ]:
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    f1_score
)

# Load the saved Epoch 5 model
model.load_state_dict(
    torch.load(EPOCH5_PATH, map_location=device, weights_only=True)
)
model.eval()


def get_video_predictions(loader, dataset):

    probabilities = []
    labels = []

    with torch.no_grad():
        for images, batch_labels in loader:

            images = images.to(device)

            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)[:, 1]

            probabilities.extend(probs.cpu().numpy())
            labels.extend(batch_labels.numpy())

    # Loader is sequential (shuffle=False)
    result = pd.DataFrame({
        "video_name": dataset.data["video_name"].values,
        "label": labels,
        "fake_probability": probabilities
    })

    # Aggregate frame predictions into one video prediction
    video_result = result.groupby(
        "video_name", as_index=False
    ).agg(
        label=("label", "first"),
        fake_probability=("fake_probability", "mean"),
        frame_count=("label", "size")
    )

    return video_result


# Generate validation and test video predictions
val_videos = get_video_predictions(val_loader, val_dataset)
test_videos = get_video_predictions(test_loader, test_dataset)

# Select threshold using validation videos only
thresholds = np.arange(0.05, 0.96, 0.01)

threshold_results = []

for threshold in thresholds:

    pred = (
        val_videos["fake_probability"].values >= threshold
    ).astype(int)

    score = balanced_accuracy_score(
        val_videos["label"], pred
    )

    threshold_results.append((threshold, score))

video_threshold, val_score = max(
    threshold_results, key=lambda x: x[1]
)

# Evaluate test videos using validation-selected threshold
test_labels = test_videos["label"].values
test_probs = test_videos["fake_probability"].values

test_predictions = (
    test_probs >= video_threshold
).astype(int)

print("Number of validation videos:", len(val_videos))
print("Number of test videos:", len(test_videos))

print("\nValidation-selected video threshold:",
      round(video_threshold, 2))

print("Validation video balanced accuracy:",
      round(val_score, 4))

print("\nVIDEO-LEVEL TEST RESULTS")
print("ROC-AUC:", round(
    roc_auc_score(test_labels, test_probs), 4
))

print("Balanced accuracy:", round(
    balanced_accuracy_score(test_labels, test_predictions), 4
))

print("Macro F1:", round(
    f1_score(test_labels, test_predictions, average="macro"), 4
))

print("\nConfusion Matrix:")
print(confusion_matrix(test_labels, test_predictions))

print("\nClassification Report:")
print(classification_report(
    test_labels,
    test_predictions,
    target_names=["REAL", "FAKE"],
    zero_division=0
))

print("\nFrames per video:")
print(test_videos["frame_count"].value_counts())

In [ ]:
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    f1_score
)

model.load_state_dict(
    torch.load(EPOCH5_PATH, map_location=device, weights_only=True)
)
model.eval()


def get_video_predictions(loader, dataset):

    probabilities = []
    labels = []

    with torch.no_grad():
        for images, batch_labels in loader:

            images = images.to(device)
            outputs = model(images)

            probs = torch.softmax(outputs, dim=1)[:, 1]

            probabilities.extend(probs.cpu().numpy())
            labels.extend(batch_labels.numpy())

    result = pd.DataFrame({
        "video_name": dataset.data["video_name"].values,
        "manipulation": dataset.data["manipulation"].values,
        "label": labels,
        "fake_probability": probabilities
    })

    # Keep different manipulation versions separate
    video_result = result.groupby(
        ["manipulation", "video_name"],
        as_index=False
    ).agg(
        label=("label", "first"),
        fake_probability=("fake_probability", "mean"),
        frame_count=("label", "size")
    )

    return video_result


# Recalculate validation and test predictions
val_videos = get_video_predictions(val_loader, val_dataset)
test_videos = get_video_predictions(test_loader, test_dataset)

print("Validation videos:", len(val_videos))
print("Test videos:", len(test_videos))

print("\nTest videos by class:")
print(test_videos.groupby("label").size())

print("\nTest videos by manipulation:")
print(test_videos.groupby("manipulation").size())

print("\nFrames per video:")
print(test_videos["frame_count"].value_counts())

# Select threshold ONLY using validation videos
thresholds = np.arange(0.05, 0.96, 0.01)

threshold_results = []

for threshold in thresholds:

    predictions = (
        val_videos["fake_probability"].values >= threshold
    ).astype(int)

    score = balanced_accuracy_score(
        val_videos["label"], predictions
    )

    threshold_results.append((threshold, score))

video_threshold, val_score = max(
    threshold_results, key=lambda x: x[1]
)

# Evaluate test videos using validation-selected threshold
y_true = test_videos["label"].values
y_prob = test_videos["fake_probability"].values

y_pred = (y_prob >= video_threshold).astype(int)

print("\nVALIDATION-SELECTED THRESHOLD:",
      round(video_threshold, 2))

print("Validation balanced accuracy:",
      round(val_score, 4))

print("\nCORRECTED VIDEO-LEVEL TEST RESULTS")

print("ROC-AUC:",
      round(roc_auc_score(y_true, y_prob), 4))

print("Balanced accuracy:",
      round(balanced_accuracy_score(y_true, y_pred), 4))

print("Macro F1:",
      round(f1_score(y_true, y_pred, average="macro"), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["REAL", "FAKE"],
    zero_division=0
))

In [ ]:
class TRADD(nn.Module):

    def __init__(self, num_classes=2, use_dct=True):

        super().__init__()

        self.use_dct = use_dct

        # Spatial branch
        backbone = models.mobilenet_v3_small(
            weights=models.MobileNet_V3_Small_Weights.DEFAULT
        )

        self.spatial_features = backbone.features
        self.spatial_pool = backbone.avgpool

        # ImageNet normalization
        self.register_buffer(
            "mean",
            torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        )

        self.register_buffer(
            "std",
            torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        )

        # DCT matrix
        self.register_buffer("dct_matrix", create_dct_matrix())

        # Frequency branch
        self.frequency_branch = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1)
        )

        # Identical classifier for both configurations
        self.classifier = nn.Sequential(
            nn.Linear(576 + 128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def extract_frequency(self, x):

        x = (x * self.std + self.mean).clamp(0, 1)

        gray = (
            0.299 * x[:, 0:1] +
            0.587 * x[:, 1:2] +
            0.114 * x[:, 2:3]
        )

        blocks = F.unfold(
            gray, kernel_size=8, stride=8
        )

        batch_size = x.shape[0]

        blocks = blocks.transpose(1, 2).reshape(
            batch_size, -1, 8, 8
        )

        dct = torch.einsum(
            "ij,bnjk,lk->bnil",
            self.dct_matrix,
            blocks,
            self.dct_matrix
        )

        frequency = dct.reshape(
            batch_size, -1, 64
        ).transpose(1, 2).reshape(
            batch_size, 64, 28, 28
        )

        frequency = (
            torch.sign(frequency) *
            torch.log1p(torch.abs(frequency))
        )

        return frequency

    def forward(self, x):

        # Spatial features
        spatial = self.spatial_features(x)
        spatial = self.spatial_pool(spatial)
        spatial = torch.flatten(spatial, 1)

        # Frequency features or zero vector
        if self.use_dct:

            frequency = self.extract_frequency(x)
            frequency = self.frequency_branch(frequency)
            frequency = torch.flatten(frequency, 1)

        else:

            frequency = torch.zeros(
                x.size(0), 128,
                device=x.device,
                dtype=spatial.dtype
            )

        # Feature fusion
        combined = torch.cat(
            [spatial, frequency], dim=1
        )

        return self.classifier(combined)

In [ ]:
rgb_model = TRADD(use_dct=False).to(device)
hybrid_model = TRADD(use_dct=True).to(device)

images, labels = next(iter(train_loader))
images = images.to(device)

rgb_model.eval()
hybrid_model.eval()

with torch.no_grad():
    rgb_output = rgb_model(images)
    hybrid_output = hybrid_model(images)

print("RGB-only output:", rgb_output.shape)
print("RGB + DCT output:", hybrid_output.shape)

print(
    "RGB parameters:",
    sum(p.numel() for p in rgb_model.parameters())
)

print(
    "Hybrid parameters:",
    sum(p.numel() for p in hybrid_model.parameters())
)

## Dataset and DCT diagnostics


In [ ]:
import os
import pandas as pd
from collections import Counter

ROOT = "/content/drive/MyDrive/TRADD"

FRAME_DIR = ROOT + "/datasets/processed/frames_expansion"
FACE_DIR = ROOT + "/datasets/processed/faces_expansion"
VIDEO_DIR = ROOT + "/datasets/raw/FaceForensicsPP_expansion"

# 1. Check extracted frame formats
frame_extensions = Counter()

for root, dirs, files in os.walk(FRAME_DIR):
    for file in files:
        ext = os.path.splitext(file)[1].lower()
        if ext:
            frame_extensions[ext] += 1

print("FRAME FORMATS")
print(frame_extensions)
print("Total frames:", sum(frame_extensions.values()))

# 2. Check face crop formats
face_extensions = Counter()

for root, dirs, files in os.walk(FACE_DIR):
    for file in files:
        ext = os.path.splitext(file)[1].lower()
        if ext in [".jpg", ".jpeg", ".png", ".webp"]:
            face_extensions[ext] += 1

print("\nFACE CROP FORMATS")
print(face_extensions)
print("Total face crops:", sum(face_extensions.values()))

# 3. Check metadata
metadata_path = FACE_DIR + "/face_metadata.csv"
metadata = pd.read_csv(metadata_path)

print("\nMETADATA")
print("Rows:", len(metadata))
print("Columns:", metadata.columns.tolist())

if "manipulation" in metadata.columns:
    print("\nFrames by manipulation:")
    print(metadata["manipulation"].value_counts())

# 4. Check ffprobe availability
print("\nFFPROBE")
print("Available:", os.system("which ffprobe > /dev/null 2>&1") == 0)

In [ ]:
import os
import json
import subprocess
import pandas as pd
from pathlib import Path

VIDEO_DIR = ROOT + "/datasets/raw/FaceForensicsPP_expansion"
OUTPUT_CSV = ROOT + "/datasets/processed/video_encoding_audit.csv"

# Find all source videos
video_paths = list(Path(VIDEO_DIR).rglob("*.mp4"))

print("Videos found:", len(video_paths))

records = []

for i, path in enumerate(video_paths):
    command = [
        "ffprobe", "-v", "error",
        "-select_streams", "v:0",
        "-show_entries",
        "stream=codec_name,width,height,pix_fmt,r_frame_rate,bit_rate",
        "-show_entries",
        "format=duration,size,bit_rate",
        "-of", "json",
        str(path)
    ]

    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=20
        )

        info = json.loads(result.stdout)
        stream = info.get("streams", [{}])[0]
        fmt = info.get("format", {})

        parts = path.parts
        category = next(
            (x for x in parts if x in [
                "original_sequences",
                "Deepfakes",
                "Face2Face",
                "FaceSwap",
                "NeuralTextures"
            ]),
            "Unknown"
        )

        if category == "original_sequences":
            category = "Original"

        records.append({
            "video": path.name,
            "category": category,
            "codec": stream.get("codec_name"),
            "width": stream.get("width"),
            "height": stream.get("height"),
            "pixel_format": stream.get("pix_fmt"),
            "fps": stream.get("r_frame_rate"),
            "duration": float(fmt.get("duration", 0) or 0),
            "file_size_MB": round(
                int(fmt.get("size", 0) or 0) / (1024**2), 3
            ),
            "bitrate_kbps": round(
                int(stream.get("bit_rate") or fmt.get("bit_rate") or 0)
                / 1000, 2
            ),
            "error": None
        })

    except Exception as e:
        records.append({
            "video": path.name,
            "category": category,
            "error": str(e)
        })

    if (i + 1) % 50 == 0:
        print(f"Processed {i+1}/{len(video_paths)} videos")

audit = pd.DataFrame(records)
audit.to_csv(OUTPUT_CSV, index=False)

print("\nSaved audit:", OUTPUT_CSV)
print("Successful:", audit["error"].isna().sum())
print("Failed:", audit["error"].notna().sum())

# Summarise technical properties by category
for column in ["codec", "width", "height", "pixel_format", "fps"]:
    print(f"\n{column.upper()} DISTRIBUTION")
    print(pd.crosstab(audit["category"], audit[column]))

print("\nNUMERIC STATISTICS BY CATEGORY")
print(
    audit.groupby("category")[
        ["duration", "file_size_MB", "bitrate_kbps"]
    ].agg(["count", "mean", "median", "std"]).round(2)
)

In [ ]:
import os
import numpy as np
import pandas as pd

audit_path = ROOT + "/datasets/processed/video_encoding_audit.csv"
audit = pd.read_csv(audit_path)

# Create a lookup table for original videos
originals = audit[audit["category"] == "Original"].copy()

originals["video_id"] = originals["video"].str.replace(
    ".mp4", "", regex=False
)

original_lookup = originals.set_index("video_id").to_dict("index")

# Compare each manipulated video with its two original IDs
comparisons = []

fake_data = audit[audit["category"] != "Original"]

for _, row in fake_data.iterrows():
    ids = row["video"].replace(".mp4", "").split("_")

    if len(ids) != 2:
        continue

    original_a = original_lookup.get(ids[0])
    original_b = original_lookup.get(ids[1])

    for position, original in [("ID_A", original_a), ("ID_B", original_b)]:
        if original is None:
            continue

        comparisons.append({
            "category": row["category"],
            "video": row["video"],
            "original_id": position,
            "width_difference": abs(
                row["width"] - original["width"]
            ),
            "height_difference": abs(
                row["height"] - original["height"]
            ),
            "duration_difference": abs(
                row["duration"] - original["duration"]
            ),
            "bitrate_difference": abs(
                row["bitrate_kbps"] - original["bitrate_kbps"]
            )
        })

comparison_df = pd.DataFrame(comparisons)

print("COMPARISON RECORDS:", len(comparison_df))

print("\nMEDIAN DIFFERENCES BY MANIPULATION")
print(
    comparison_df.groupby(["category", "original_id"])[
        ["width_difference", "height_difference",
         "duration_difference", "bitrate_difference"]
    ].median().round(2)
)

print("\nEXACT RESOLUTION MATCH RATE (%)")
comparison_df["resolution_match"] = (
    (comparison_df["width_difference"] == 0) &
    (comparison_df["height_difference"] == 0)
)

print(
    comparison_df.groupby(["category", "original_id"])[
        "resolution_match"
    ].mean().mul(100).round(1)
)

print("\nEXACT DURATION MATCH RATE (%)")
comparison_df["duration_match"] = (
    comparison_df["duration_difference"] < 0.05
)

print(
    comparison_df.groupby(["category", "original_id"])[
        "duration_match"
    ].mean().mul(100).round(1)
)

In [ ]:
from PIL import Image
import os
import numpy as np
import pandas as pd

metadata = pd.read_csv(
    ROOT + "/datasets/processed/faces_expansion/face_metadata.csv"
)

results = []

for _, row in metadata.iterrows():
    try:
        # Inspect original extracted frame
        with Image.open(row["frame_path"]) as img:
            frame_width, frame_height = img.size

        # Inspect saved face crop
        with Image.open(row["face_path"]) as img:
            crop_width, crop_height = img.size

        crop_size_kb = os.path.getsize(row["face_path"]) / 1024
        frame_size_kb = os.path.getsize(row["frame_path"]) / 1024

        results.append({
            "manipulation": row["manipulation"],
            "label": row["label"],
            "frame_width": frame_width,
            "frame_height": frame_height,
            "crop_width": crop_width,
            "crop_height": crop_height,
            "crop_size_kb": crop_size_kb,
            "frame_size_kb": frame_size_kb
        })

    except Exception as e:
        results.append({
            "manipulation": row["manipulation"],
            "label": row["label"],
            "error": str(e)
        })

crop_audit = pd.DataFrame(results)
if "error" not in crop_audit.columns:
    crop_audit["error"] = None

valid = crop_audit[crop_audit["error"].isna()].copy()

print("TOTAL RECORDS:", len(crop_audit))
print("VALID:", len(valid))
print("FAILED:", len(crop_audit) - len(valid))

print("\nCROP DIMENSIONS")
print(
    valid.groupby("manipulation")[
        ["crop_width", "crop_height"]
    ].agg(["min", "max", "nunique"])
)

print("\nSOURCE FRAME DIMENSIONS")
print(
    valid.groupby("manipulation")[
        ["frame_width", "frame_height"]
    ].agg(["min", "max", "nunique"])
)

print("\nFILE SIZE STATISTICS (KB)")
print(
    valid.groupby("manipulation")[
        ["crop_size_kb", "frame_size_kb"]
    ].agg(["mean", "median", "std", "min", "max"]).round(2)
)

In [ ]:
import torch
import inspect

# 1. Check DCT matrix
dct = create_dct_matrix(8)

print("DCT MATRIX SHAPE:", dct.shape)

identity = dct @ dct.T

print(
    "ORTHOGONALITY ERROR:",
    torch.max(torch.abs(identity - torch.eye(8))).item()
)

# 2. Inspect the actual extraction function
print("\nDCT EXTRACTION FUNCTION:\n")
print(inspect.getsource(TRADD.extract_frequency))

# 3. Check frequency output on one real batch
batch_images, batch_labels = next(iter(train_loader))

hybrid_model.eval()

with torch.no_grad():
    frequency_features = hybrid_model.extract_frequency(
        batch_images.to(device)
    )

print("\nFREQUENCY OUTPUT")
print("Shape:", frequency_features.shape)
print("Minimum:", frequency_features.min().item())
print("Maximum:", frequency_features.max().item())
print("Mean:", frequency_features.mean().item())
print("Standard deviation:", frequency_features.std().item())
print("NaN values:", torch.isnan(frequency_features).sum().item())
print("Infinite values:", torch.isinf(frequency_features).sum().item())

In [ ]:
import matplotlib.pyplot as plt
import torch

hybrid_model.eval()

# One sample from the batch
sample = batch_images[0:1].to(device)

with torch.no_grad():
    freq = hybrid_model.extract_frequency(sample)

# Select coefficient channels
channels = [0, 1, 2, 8, 16, 24, 32, 40, 48, 56, 63]

fig, axes = plt.subplots(3, 4, figsize=(14, 10))

# Convert normalized RGB back to displayable image
img = (
    sample[0].cpu() * hybrid_model.std.cpu()
    + hybrid_model.mean.cpu()
).clamp(0, 1)

img = img[0].permute(1, 2, 0).numpy()

axes[0, 0].imshow(img)
axes[0, 0].set_title("Original Face")

for ax, channel in zip(axes.flat[1:], channels):
    feature_map = freq[0, channel].cpu().numpy()
    ax.imshow(feature_map, cmap="seismic")
    ax.set_title(f"DCT coefficient {channel}")

for ax in axes.flat:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import inspect

print(inspect.getsource(create_dct_matrix))

print("\nDCT MATRIX:")
print(create_dct_matrix(8))

In [ ]:
import pandas as pd

print("=== EXISTING DATAFRAMES ===")

dataframes = {}

for name, obj in globals().copy().items():
    if isinstance(obj, pd.DataFrame):
        dataframes[name] = obj
        print(f"\nVariable: {name}")
        print("Shape:", obj.shape)
        print("Columns:", obj.columns.tolist())

print("\n=== MODEL AND LOADER VARIABLES ===")

for name in [
    "model", "hybrid_model", "rgb_model",
    "test_loader", "test_dataset",
    "metadata", "device"
]:
    print(name, ":", name in globals())

In [ ]:
required_columns = {
    "manipulation",
    "label",
    "fake_probability"
}

candidates = {
    name: df
    for name, df in dataframes.items()
    if required_columns.issubset(df.columns)
}

print("Matching prediction DataFrames:", list(candidates.keys()))

if len(candidates) == 1:
    results = next(iter(candidates.values())).copy()
    print("\nPrediction DataFrame found!")
    print("Rows:", len(results))
    print(results.head())

elif len(candidates) == 0:
    print("\nNo existing video prediction DataFrame found.")
    print("We will need to reconstruct predictions from the saved checkpoint.")

else:
    print("\nMultiple matching DataFrames found.")
    print("Please share their names so we can select the correct test predictions.")

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    recall_score,
    f1_score
)
import pandas as pd

results = test_videos.copy()
threshold = 0.41

rows = []

for method in [
    "Deepfakes",
    "Face2Face",
    "FaceSwap",
    "NeuralTextures"
]:
    subset = results[
        results["manipulation"].isin(["Original", method])
    ].copy()

    y_true = subset["label"].astype(int)
    y_prob = subset["fake_probability"]
    y_pred = (y_prob >= threshold).astype(int)

    rows.append({
        "Manipulation": method,
        "REAL videos": (y_true == 0).sum(),
        "FAKE videos": (y_true == 1).sum(),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "Balanced Accuracy": balanced_accuracy_score(
            y_true, y_pred
        ),
        "REAL Recall": recall_score(
            y_true, y_pred, pos_label=0
        ),
        "FAKE Recall": recall_score(
            y_true, y_pred, pos_label=1
        ),
        "Macro F1": f1_score(
            y_true, y_pred, average="macro"
        )
    })

method_results = pd.DataFrame(rows)

print(method_results.round(4).to_string(index=False))

## Final training, temporal modelling and reliability evaluation


In [ ]:
required = [
    "TRADD",
    "train_loader",
    "val_loader",
    "train_data",
    "val_data",
    "device",
    "set_seed"
]

for name in required:
    print(f"{name}: {'Available' if name in globals() else 'MISSING'}")

print("\nMODEL CHECK")

print("RGB model:", type(rgb_model).__name__)
print("Hybrid model:", type(hybrid_model).__name__)

print("\nDATASET SIZES")
print("Training:", len(train_data))
print("Validation:", len(val_data))

print("\nDEVICE:", device)

print("\nMODEL CONFIGURATION")
print("RGB DCT setting:", rgb_model.use_dct)
print("Hybrid DCT setting:", hybrid_model.use_dct)

In [ ]:
import random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# Inspect one training batch
batch = next(iter(train_loader))

print("Batch type:", type(batch))

if isinstance(batch, (list, tuple)):
    print("Number of batch elements:", len(batch))

    for i, item in enumerate(batch):
        if torch.is_tensor(item):
            print(f"Element {i}: shape={item.shape}, dtype={item.dtype}")
        else:
            print(f"Element {i}: type={type(item)}")

print("\nDataset type:", type(train_loader.dataset))
print("Dataset attributes:", train_loader.dataset.__dict__.keys())

In [ ]:
print("VALIDATION DATA COLUMNS")
print(val_loader.dataset.data.columns.tolist())

print("\nFIRST 5 VALIDATION ROWS")
display(val_loader.dataset.data.head())

print("\nVALIDATION LOADER")
print("Batch size:", val_loader.batch_size)
print("Sampler:", type(val_loader.sampler).__name__)

print("\nMODEL OUTPUT TEST")
images, labels = next(iter(val_loader))

with torch.no_grad():
    output = hybrid_model(images.to(device))

print("Input shape:", images.shape)
print("Output shape:", output.shape)
print("Labels shape:", labels.shape)

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
from sklearn.metrics import roc_auc_score

model = TRADD(use_dct=False).to(device)

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor([2.5, 0.625], device=device)
)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=1e-4, weight_decay=1e-4
)

best_auc = 0
patience = 0
history = []

for epoch in range(15):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    model.eval()
    predictions = []

    with torch.no_grad():
        for images, labels in val_loader:
            probabilities = torch.softmax(
                model(images.to(device)), dim=1
            )[:, 1]
            predictions.extend(probabilities.cpu().numpy())

    val = val_loader.dataset.data.copy()
    val["probability"] = predictions

    videos = val.groupby(
        ["manipulation", "video_name"], as_index=False
    ).agg(label=("label", "first"),
          probability=("probability", "mean"))

    auc = roc_auc_score(videos["label"], videos["probability"])
    train_loss = total_loss / len(train_loader)

    history.append([epoch + 1, train_loss, auc])

    print(f"Epoch {epoch+1}: Loss={train_loss:.4f}, Video AUC={auc:.4f}")

    if auc > best_auc:
        best_auc = auc
        patience = 0
        torch.save(model.state_dict(),
                   "/content/drive/MyDrive/TRADD/models/rgb_best.pth")
    else:
        patience += 1

    if patience == 4:
        break

print("Best validation video AUC:", best_auc)

> Original cell 128 was empty or unavailable in the exported source.


In [ ]:
model = TRADD(use_dct=True).to(device)

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor([2.5, 0.625], device=device)
)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=1e-4, weight_decay=1e-4
)

best_auc = 0
patience = 0
hybrid_history = []

for epoch in range(15):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    model.eval()
    predictions = []

    with torch.no_grad():
        for images, labels in val_loader:
            probabilities = torch.softmax(
                model(images.to(device)), dim=1
            )[:, 1]
            predictions.extend(probabilities.cpu().numpy())

    val = val_loader.dataset.data.copy()
    val["probability"] = predictions

    videos = val.groupby(
        ["manipulation", "video_name"], as_index=False
    ).agg(label=("label", "first"),
          probability=("probability", "mean"))

    auc = roc_auc_score(videos["label"], videos["probability"])
    train_loss = total_loss / len(train_loader)

    hybrid_history.append([epoch + 1, train_loss, auc])

    print(f"Epoch {epoch+1}: Loss={train_loss:.4f}, Video AUC={auc:.4f}")

    if auc > best_auc:
        best_auc = auc
        patience = 0
        torch.save(
            model.state_dict(),
            "/content/drive/MyDrive/TRADD/models/hybrid_best.pth"
        )
    else:
        patience += 1

    if patience == 4:
        break

print("Best hybrid validation video AUC:", best_auc)

In [ ]:
# Load best checkpoints
rgb_model = TRADD(use_dct=False).to(device)
rgb_model.load_state_dict(torch.load(
    "/content/drive/MyDrive/TRADD/models/rgb_best.pth",
    map_location=device,
    weights_only=True
))

hybrid_model = TRADD(use_dct=True).to(device)
hybrid_model.load_state_dict(torch.load(
    "/content/drive/MyDrive/TRADD/models/hybrid_best.pth",
    map_location=device,
    weights_only=True
))

def predict_videos(model):
    model.eval()
    predictions = []

    with torch.no_grad():
        for images, labels in test_loader:
            probs = torch.softmax(model(images.to(device)), dim=1)[:, 1]
            predictions.extend(probs.cpu().numpy())

    result = test_loader.dataset.data.copy()
    result["probability"] = predictions

    return result.groupby(
        ["manipulation", "video_name"], as_index=False
    ).agg(label=("label", "first"),
          probability=("probability", "mean"))

rgb_test = predict_videos(rgb_model)
hybrid_test = predict_videos(hybrid_model)

print("RGB Test AUC:", roc_auc_score(
    rgb_test["label"], rgb_test["probability"]
))

print("Hybrid Test AUC:", roc_auc_score(
    hybrid_test["label"], hybrid_test["probability"]
))

In [ ]:
from sklearn.metrics import (
    balanced_accuracy_score, accuracy_score,
    precision_score, recall_score, f1_score,
    confusion_matrix
)
import numpy as np

def evaluate_model(model):
    def get_predictions(loader):
        model.eval()
        probs = []

        with torch.no_grad():
            for images, labels in loader:
                output = model(images.to(device))
                probs.extend(
                    torch.softmax(output, dim=1)[:, 1].cpu().numpy()
                )

        return np.array(probs)

    val_probs = get_predictions(val_loader)
    test_probs = get_predictions(test_loader)

    val_meta = val_loader.dataset.data
    test_meta = test_loader.dataset.data

    def aggregate(meta, probs):
        df = meta.copy()
        df["probability"] = probs
        return df.groupby(
            ["manipulation", "video_name"], as_index=False
        ).agg(label=("label", "first"),
              probability=("probability", "mean"))

    val = aggregate(val_meta, val_probs)
    test = aggregate(test_meta, test_probs)

    thresholds = np.arange(0.05, 0.96, 0.01)
    threshold = max(
        thresholds,
        key=lambda t: balanced_accuracy_score(
            val["label"], (val["probability"] >= t).astype(int)
        )
    )

    y_true = test["label"]
    y_pred = (test["probability"] >= threshold).astype(int)

    print("Threshold:", round(threshold, 2))
    print("Test AUC:", round(roc_auc_score(y_true, test["probability"]), 4))
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print("Balanced Accuracy:", round(balanced_accuracy_score(y_true, y_pred), 4))
    print("Precision:", round(precision_score(y_true, y_pred), 4))
    print("Recall:", round(recall_score(y_true, y_pred), 4))
    print("Macro F1:", round(f1_score(y_true, y_pred, average="macro"), 4))
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

print("RGB-ONLY RESULTS")
evaluate_model(rgb_model)

print("\nRGB + DCT RESULTS")
evaluate_model(hybrid_model)

In [ ]:
print(model)

In [ ]:
model = hybrid_model
model.eval()

images, labels = next(iter(train_loader))
images = images.to(device)

with torch.no_grad():
    spatial = model.spatial_features(images)
    spatial = model.spatial_pool(spatial).flatten(1)

    frequency = model.extract_frequency(images)
    frequency = model.frequency_branch(frequency).flatten(1)

    features = torch.cat([spatial, frequency], dim=1)

print("Spatial:", spatial.shape)
print("Frequency:", frequency.shape)
print("Fused:", features.shape)

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torch
import pandas as pd

class VideoSequenceDataset(Dataset):
    def __init__(self, data, transform):
        self.data = data
        self.transform = transform

        self.groups = [
            group.sort_values("frame_number")
            for _, group in data.groupby(
                ["manipulation", "video_name"]
            )
        ]

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        group = self.groups[idx]

        frames = []
        for path in group["face_path"]:
            image = Image.open(path).convert("RGB")
            frames.append(self.transform(image))

        return torch.stack(frames), int(group["label"].iloc[0])


transform = val_loader.dataset.transform

train_sequences = VideoSequenceDataset(train_data, transform)
val_sequences = VideoSequenceDataset(val_data, transform)
test_sequences = VideoSequenceDataset(test_data, transform)

train_video_loader = DataLoader(train_sequences, batch_size=4, shuffle=True)
val_video_loader = DataLoader(val_sequences, batch_size=4, shuffle=False)
test_video_loader = DataLoader(test_sequences, batch_size=4, shuffle=False)

print("Training videos:", len(train_sequences))
print("Validation videos:", len(val_sequences))
print("Test videos:", len(test_sequences))

frames, labels = next(iter(train_video_loader))
print("Sequence shape:", frames.shape)

In [ ]:
def extract_video_features(loader, model):
    model.eval()
    all_features, all_labels = [], []

    with torch.no_grad():
        for frames, labels in loader:
            b, t, c, h, w = frames.shape
            frames = frames.reshape(b * t, c, h, w).to(device)

            spatial = model.spatial_pool(
                model.spatial_features(frames)
            ).flatten(1)

            frequency = model.frequency_branch(
                model.extract_frequency(frames)
            ).flatten(1)

            fused = torch.cat([spatial, frequency], dim=1)
            all_features.append(fused.reshape(b, t, 704).cpu())
            all_labels.append(labels)

    return torch.cat(all_features), torch.cat(all_labels)


train_features, train_labels = extract_video_features(
    train_video_loader, hybrid_model
)

val_features, val_labels = extract_video_features(
    val_video_loader, hybrid_model
)

test_features, test_labels = extract_video_features(
    test_video_loader, hybrid_model
)

print("Training features:", train_features.shape)
print("Validation features:", val_features.shape)
print("Test features:", test_features.shape)

In [ ]:
import torch.nn as nn

class TemporalTRADD(nn.Module):
    def __init__(self, input_size=704, hidden_size=128):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True,
            num_layers=1
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        _, hidden = self.gru(x)
        return self.classifier(hidden[-1])


temporal_model = TemporalTRADD().to(device)

print(temporal_model)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_feature_loader = DataLoader(
    TensorDataset(train_features, train_labels),
    batch_size=16, shuffle=True
)

val_feature_loader = DataLoader(
    TensorDataset(val_features, val_labels),
    batch_size=16, shuffle=False
)

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor([2.5, 0.625], device=device)
)

optimizer = torch.optim.AdamW(
    temporal_model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

best_auc = 0
patience = 0

for epoch in range(30):
    temporal_model.train()
    total_loss = 0

    for x, y in train_feature_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        loss = criterion(temporal_model(x), y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    temporal_model.eval()
    predictions, actual = [], []

    with torch.no_grad():
        for x, y in val_feature_loader:
            probs = torch.softmax(
                temporal_model(x.to(device)), dim=1
            )[:, 1]

            predictions.extend(probs.cpu().numpy())
            actual.extend(y.numpy())

    auc = roc_auc_score(actual, predictions)

    print(
        f"Epoch {epoch+1}: "
        f"Loss={total_loss/len(train_feature_loader):.4f}, "
        f"Val AUC={auc:.4f}"
    )

    if auc > best_auc:
        best_auc = auc
        patience = 0
        torch.save(
            temporal_model.state_dict(),
            "/content/drive/MyDrive/TRADD/models/tradd_temporal_best.pth"
        )
    else:
        patience += 1

    if patience == 5:
        break

print("Best temporal validation AUC:", best_auc)

In [ ]:
temporal_model.load_state_dict(torch.load(
    "/content/drive/MyDrive/TRADD/models/tradd_temporal_best.pth",
    map_location=device,
    weights_only=True
))

temporal_model.eval()
predictions, actual = [], []

test_feature_loader = DataLoader(
    TensorDataset(test_features, test_labels),
    batch_size=16,
    shuffle=False
)

with torch.no_grad():
    for x, y in test_feature_loader:
        probs = torch.softmax(
            temporal_model(x.to(device)), dim=1
        )[:, 1]

        predictions.extend(probs.cpu().numpy())
        actual.extend(y.numpy())

print("Temporal Test AUC:",
      round(roc_auc_score(actual, predictions), 4))

In [ ]:
def get_probabilities(loader):
    temporal_model.eval()
    probs, labels = [], []

    with torch.no_grad():
        for x, y in loader:
            output = temporal_model(x.to(device))
            probs.extend(torch.softmax(output, dim=1)[:, 1].cpu().numpy())
            labels.extend(y.numpy())

    return np.array(probs), np.array(labels)


val_probs, val_y = get_probabilities(val_feature_loader)
test_probs, test_y = get_probabilities(test_feature_loader)

# Confidence of the predicted class
val_confidence = np.maximum(val_probs, 1 - val_probs)
val_predictions = (val_probs >= 0.5).astype(int)

# Select the lowest confidence threshold giving <=10% validation error
threshold = 1.01

for t in np.sort(np.unique(val_confidence)):
    accepted = val_confidence >= t

    if accepted.sum() > 0:
        error = np.mean(val_predictions[accepted] != val_y[accepted])

        if error <= 0.10:
            threshold = t
            break

print("Selected confidence threshold:", round(threshold, 4))

In [ ]:
accepted = test_confidence >= threshold

if accepted.sum() > 0:
    accuracy = np.mean(
        test_predictions[accepted] == test_y[accepted]
    )
else:
    accuracy = float("nan")

print("Total test videos:", len(test_y))
print("Accepted predictions:", accepted.sum())
print("Uncertain predictions:", (~accepted).sum())
print("Coverage:", round(accepted.mean(), 4))
print("Accuracy on accepted predictions:", round(accuracy, 4))

print("\nPrediction distribution:")
print("REAL:", np.sum(accepted & (test_predictions == 0)))
print("FAKE:", np.sum(accepted & (test_predictions == 1)))
print("UNCERTAIN:", np.sum(~accepted))

In [ ]:
from sklearn.metrics import confusion_matrix

print("Accepted prediction confusion matrix:")
print(confusion_matrix(
    test_y[accepted],
    test_predictions[accepted],
    labels=[0, 1]
))

for label, name in [(0, "REAL"), (1, "FAKE")]:
    mask = test_y == label
    print(
        f"{name}: {np.sum(accepted & mask)}/{np.sum(mask)} "
        f"accepted"
    )

In [ ]:
def find_threshold(probs, labels, target_class, min_precision=0.90):
    candidates = np.sort(np.unique(probs))

    best_threshold = None
    best_coverage = 0

    for threshold in candidates:
        if target_class == 0:
            accepted = probs <= threshold
        else:
            accepted = probs >= threshold

        if accepted.sum() < 5:
            continue

        precision = np.mean(labels[accepted] == target_class)

        if precision >= min_precision and accepted.sum() > best_coverage:
            best_threshold = threshold
            best_coverage = accepted.sum()

    return best_threshold


real_threshold = find_threshold(
    val_probs, val_y, target_class=0
)

fake_threshold = find_threshold(
    val_probs, val_y, target_class=1
)

print("REAL threshold:", real_threshold)
print("FAKE threshold:", fake_threshold)

In [ ]:
real_accepted = test_probs <= real_threshold
fake_accepted = test_probs >= fake_threshold

accepted = real_accepted | fake_accepted

final_predictions = np.full(len(test_y), -1)
final_predictions[real_accepted] = 0
final_predictions[fake_accepted] = 1

correct = final_predictions[accepted] == test_y[accepted]

print("Total videos:", len(test_y))
print("REAL predictions:", real_accepted.sum())
print("FAKE predictions:", fake_accepted.sum())
print("UNCERTAIN predictions:", (~accepted).sum())

print("Coverage:", round(accepted.mean(), 4))
print("Selective accuracy:", round(correct.mean(), 4) if accepted.any() else "N/A")

In [ ]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(
    test_y[accepted],
    final_predictions[accepted],
    labels=[0, 1]
))

In [ ]:
import pandas as pd
import os

results = pd.DataFrame([
    ["RGB-only", 0.8294, 0.820625],
    ["RGB + DCT", 0.8038, 0.830625],
    ["RGB + DCT + GRU", 0.8044, 0.8225]
], columns=["Model", "Validation_AUC", "Test_AUC"])

save_path = "/content/drive/MyDrive/TRADD/results"
os.makedirs(save_path, exist_ok=True)

results.to_csv(
    f"{save_path}/model_comparison.csv",
    index=False
)

reliability = pd.DataFrame([{
    "REAL_threshold": real_threshold,
    "FAKE_threshold": fake_threshold,
    "Coverage": 0.75,
    "Selective_accuracy": 0.9067,
    "Uncertain_videos": 25
}])

reliability.to_csv(
    f"{save_path}/reliability_results.csv",
    index=False
)

print("Experimental results saved successfully.")

## Final model comparison


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = ["RGB", "RGB + DCT", "RGB + DCT + GRU"]
val_auc = [0.8294, 0.8038, 0.8044]
test_auc = [0.8206, 0.8306, 0.8225]

x = np.arange(len(models))
width = 0.35

plt.figure(figsize=(9, 5))

plt.bar(x - width/2, val_auc, width, label="Validation AUC")
plt.bar(x + width/2, test_auc, width, label="Test AUC")

plt.ylabel("ROC-AUC")
plt.xlabel("Model")
plt.title("TRADD Model Performance Comparison")
plt.xticks(x, models)
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()

path = "/content/drive/MyDrive/TRADD/results/model_comparison.png"
plt.savefig(path, dpi=300)
plt.show()

print("Graph saved:", path)